# MLOps Example Notebook - Training from an uploaded dataset

This notebook shows the **complete dataset flow** of the platform:

1. The user uploads a dataset from the UI. It is stored in **MinIO**
   (bucket `datasets`) and associated with the repository. Only one dataset is
   **active** per repository at a time.
2. When a pipeline runs, the worker downloads the active dataset from MinIO and
   injects its local path into the notebook as the papermill parameter
   **`DATASET_PATH`**.
3. The notebook reads that path, trains a model, logs metrics to MLflow and
   exports the artifact to `MODEL_OUTPUT_PATH`.

If `DATASET_PATH` is empty (the notebook is run by hand, or the repository has
no active dataset) it falls back to a small synthetic dataset, so the notebook
always runs end to end.

**Required tags:** `mlops:config`, `mlops:preprocessing`, `mlops:training`, `mlops:export`

**Optional tags:** `mlops:data`, `mlops:evaluation`, `parameters`

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.

DATASET_PATH = ""           # Local path of the active dataset downloaded from MinIO
MODEL_OUTPUT_PATH = ""      # Where the trained model must be saved
PIPELINE_ID = ""            # UUID of the current pipeline run
MLFLOW_TRACKING_URI = ""    # MLflow tracking server URL

In [ ]:
# ============================================================
# mlops:config - Model metadata
# ============================================================
# The platform reads MODEL_NAME and VERSION to register the model.

import os

MODEL_NAME = "dataset-demo-classifier"
VERSION = "1"

# Fallbacks so the notebook also works when executed outside the pipeline.
MODEL_OUTPUT_PATH = MODEL_OUTPUT_PATH or "./model.joblib"
PIPELINE_ID = PIPELINE_ID or "local-dev"
MLFLOW_TRACKING_URI = MLFLOW_TRACKING_URI or os.getenv(
    "MLFLOW_TRACKING_URI", "http://localhost:5000"
)

print(f"Model: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")
print(f"Dataset path: {DATASET_PATH or '(empty - synthetic fallback)'}")
print(f"Model output: {MODEL_OUTPUT_PATH}")

In [ ]:
# ============================================================
# mlops:data - Read the dataset injected by the pipeline
# ============================================================
# DATASET_PATH points at the dataset that is ACTIVE for this repository.
# The worker downloads it from MinIO (bucket "datasets") before executing the
# notebook, so here it is just a normal local file.

from pathlib import Path

import pandas as pd

TARGET_COLUMN = "target"


def load_dataset(path: str) -> pd.DataFrame:
    """Read the injected dataset. CSV, Parquet and JSON are supported."""
    suffix = Path(path).suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in (".json", ".jsonl"):
        return pd.read_json(path, lines=suffix == ".jsonl")
    return pd.read_csv(path)


if DATASET_PATH and Path(DATASET_PATH).exists():
    df = load_dataset(DATASET_PATH)
    print(f"Loaded uploaded dataset from {DATASET_PATH}")
else:
    # Standalone fallback: no dataset injected, generate one.
    from sklearn.datasets import make_classification

    features, labels = make_classification(
        n_samples=600,
        n_features=8,
        n_informative=5,
        n_classes=3,
        random_state=42,
    )
    df = pd.DataFrame(features, columns=[f"feature_{i}" for i in range(features.shape[1])])
    df[TARGET_COLUMN] = labels
    print("DATASET_PATH is empty - using a synthetic dataset instead.")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# mlops:preprocessing - Feature selection, split and scaling
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# If the uploaded dataset has no "target" column, fall back to the last column.
if TARGET_COLUMN not in df.columns:
    TARGET_COLUMN = df.columns[-1]
    print(f"No 'target' column found - using '{TARGET_COLUMN}' as the label.")

y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])

# Keep it simple and robust: numeric features only, missing values filled with 0.
X = X.select_dtypes(include="number").fillna(0)
FEATURE_NAMES = list(X.columns)

stratify = y if y.nunique() <= 20 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")
print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")

In [ ]:
# ============================================================
# mlops:training - Model training
# ============================================================
# Replace RandomForestClassifier with whatever your project needs.

from sklearn.ensemble import RandomForestClassifier

N_ESTIMATORS = 200
MAX_DEPTH = 8

model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_scaled, y_train)
print("Training complete.")

In [ ]:
# ============================================================
# mlops:evaluation - Metrics and MLflow logging
# ============================================================
# The platform reads the 'accuracy' metric for its auto-deployment decision.
# Artifacts and metrics land in MLflow, whose artifact store is the MinIO
# bucket "mlflow" (s3://mlflow/).

import mlflow
from sklearn.metrics import accuracy_score, classification_report, f1_score

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy:.4f} | F1 (weighted): {f1:.4f}")

# The pipeline also injects MLFLOW_RUN_ID and then reads the metrics back from
# that run, so log into it. globals().get() keeps the notebook runnable
# standalone, where the variable simply does not exist.
RUN_ID = globals().get("MLFLOW_RUN_ID") or ""

started_here = mlflow.active_run() is None
if started_here:
    if RUN_ID:
        mlflow.start_run(run_id=RUN_ID)
    else:
        mlflow.start_run(run_name=f"{MODEL_NAME}-{PIPELINE_ID}")

mlflow.log_param("dataset_path", DATASET_PATH or "synthetic")
mlflow.log_param("target_column", TARGET_COLUMN)
mlflow.log_metric("accuracy", accuracy)
mlflow.log_metric("f1_weighted", f1)
mlflow.log_metric("n_features", float(len(FEATURE_NAMES)))
mlflow.log_metric("n_samples", float(len(df)))

if started_here:
    mlflow.end_run()

print("Metrics logged to MLflow.")

In [ ]:
# ============================================================
# mlops:export - Save the model artifact
# ============================================================
# The platform picks this file up and registers it in MLflow, which uploads it
# to MinIO (s3://mlflow/).

import joblib

joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Model saved to {MODEL_OUTPUT_PATH}")